# nb_02 — Silver: conform, deduplicate, validate, convert

**Module 2.** One trustworthy row per workforce event.

1. **Deduplicate** replayed `event_id`s — keep the latest `ingest_ts`.
2. **Conform** the work-country code — `CORE_HR` emits ISO-3, `PAYROLL` emits ISO-2.
3. **Validate & quarantine** — negative pay, orphan employees, bad dates, unknown currency.
4. **Convert** pay amounts from local currency to **CAD** (the grid is in CAD).
5. Write `silver.workforce_event`.

> Note: many events (leaves, deployments) legitimately carry **no amount**. We
> quarantine *negative* pay, not null pay.

> **Lab notebook.** This is the fill-in-the-blank companion to the solution notebook of the same name. Each code cell only contains `# TODO` comments — use the markdown cell above each one to figure out what to build.


## Load Bronze inputs

**Summary.** Creates the `silver` schema and loads the Bronze tables this notebook conforms: raw events (including replays) plus the workers, workers-delta, and FX reference tables.


In [ ]:
# TODO: Load the Bronze inputs this notebook conforms.
# - Create the `silver` schema if it doesn't exist.
# - Load bronze.workforce_events_raw, bronze.workers, bronze.workers_delta, and
#   bronze.fx_rates into DataFrames.
# - Print the raw event count (still including replays).


## 1. Deduplicate replays

The payroll system replays events; the same `event_id` arrives again with a later
`ingest_ts`. Keep the newest per `event_id`.


## Deduplicate replays

**Summary.** Keeps only the newest copy of each `event_id`, removing the payroll system's replayed duplicates.


In [ ]:
# TODO: Deduplicate replayed events, keeping the newest ingest per event_id.
# - Build a window partitioned by event_id, ordered by ingest_ts descending.
# - Number rows within each event and keep only the top-ranked (latest) row.
# - Print the surviving row count and how many duplicates were removed.


## 2. Conform the work-country code to ISO-3

`CORE_HR` sends ISO-3 (`CAN`), `PAYROLL` sends ISO-2 (`CA`). Map everything to
ISO-3 using a small static reference (the workforce spans a fixed set of
countries).


## Conform the work-country code to ISO-3

**Summary.** Normalizes the work-country code so both source systems agree: `CORE_HR` already sends ISO-3, `PAYROLL` sends ISO-2, which a small static map converts.


In [ ]:
# TODO: Conform the work-country code to a single ISO-3 representation.
# - Build a small static ISO-2 -> ISO-3 reference for the countries in scope.
# - Keep values that are already 3 characters long (already ISO-3).
# - Left-join the reference to look up ISO-3 for ISO-2 values.
# - Coalesce the two into a single work_country_iso3 column and drop the helper columns.


## 3. Validate & quarantine

| Rule | Reason |
|------|--------|
| `amount_local < 0` | `negative_pay` |
| `employee_id` doesn't resolve | `orphan_employee` |
| `event_date` before 2021-01-01 or future | `date_out_of_range` |
| unknown currency **or** unresolved country | `unresolved_reference` |

Null amounts are **valid** (non-pay events). Failing rows → quarantine.


## Validate and quarantine

**Summary.** Flags rows that break data-quality rules (negative pay, orphan employees, out-of-range dates, unresolved currency/country), writes the failures to a quarantine table, and keeps the clean rows.


In [ ]:
# TODO: Validate rows and split off quarantine failures.
# - Collect the sets of known employee IDs (workers + workers_delta) and known currencies.
# - Flag the first matching data-quality reason for each row: negative_pay (amount < 0),
#   orphan_employee (employee_id not known), date_out_of_range (before 2021-01-01 or in
#   the future), or unresolved_reference (unknown currency or unresolved country). Null
#   amounts are valid (non-pay events), not a failure.
# - Split into quarantine (any reason set) and clean (no reason) DataFrames.
# - Overwrite silver.workforce_event_quarantine with the failures.
# - Print a summary of quarantined rows by reason and the count continuing as clean.


## 4. Convert pay to CAD

The pay grid is in CAD, but international offices pay in local currency. Convert
`amount_local` → `amount_cad` on `(rate_month, currency)`. Non-pay events keep a
null amount.


## Convert pay to CAD and write Silver

**Summary.** Converts local-currency amounts to CAD using the FX rate for the event month, selects the conformed columns, and writes the trustworthy `silver.workforce_event` table.


In [ ]:
# TODO: Convert local-currency pay to CAD and write the Silver table.
# - Build an FX lookup keyed by (rate month, currency) with a cad_per_unit rate.
# - Derive a rate_month from event_date (yyyy-MM) and left-join the FX lookup on it.
# - Compute amount_cad: keep it null when amount_local is null, otherwise multiply by the
#   matched rate (default to 1.0 if no rate is found), rounded to 2 decimals.
# - Select and cast the final set of columns and overwrite silver.workforce_event.
# - Print the row count and preview a few rows.
